In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 

In [ ]:
types = ["ask","bid","last"]
dfs = {}

for t in types:
    df = pd.read_csv(
        f"../data/raw/DAT_NT_EURUSD_T_{t.upper()}_202601.csv",
        sep=";",
        header=None,
        usecols=[0,1],
        names=["datetime", t],
        parse_dates=["datetime"],
        date_format="%Y%m%d %H%M%S"
    )
    df = df.set_index("datetime")
    df = df.loc["2026-01-05 00:00":"2026-01-05 00:10"]
    df = df[t].resample("1s").last().dropna()
    dfs[t] = df

ask, bid, last = dfs["ask"], dfs["bid"], dfs["last"]

print(ask.head())
print(bid.head())
print(last.head())


In [ ]:
df = pd.concat(dfs, axis=1)
df["mid"] = (df["ask"] + df["bid"]) / 2

fig, axes = plt.subplots(4, 1, figsize=(12, 24), sharex=True)
axes[0].plot(df.index, df["ask"], label="Ask", alpha=0.8)
axes[0].plot(df.index, df["bid"], label="Bid", alpha=0.8)
axes[0].plot(df.index, df["last"], label="Last", alpha=0.8)
axes[0].plot(df.index, df["mid"], label="Mid (Ask+Bid)/2", alpha=0.8, linestyle="--")
axes[0].set_title("EUR/USD — Ask, Bid, Last & Mid")
axes[0].set_ylabel("Price")
axes[0].legend()
for i, t in enumerate(["ask", "bid", "last"]):
    ax = axes[i + 1]
    ax.plot(df.index, df[t], label=t.capitalize(), color=f"C{i}")
    ax.set_title(f"EUR/USD — {t.capitalize()}")
    ax.set_ylabel("Price")
plt.tight_layout()
fig.savefig("../plots/eda/eurusd_ask_bid_last.pdf", bbox_inches="tight")
print("Saved!")
plt.show()

In [ ]:
types = ["ask", "bid", "last"]
dfs = {}

for t in types:
    df = pd.read_csv(
        f"../data/raw/DAT_NT_EURUSD_T_{t.upper()}_202601.csv",
        sep=";",
        header=None,
        usecols=[0, 1],
        names=["datetime", t],
        parse_dates=["datetime"],
        date_format="%Y%m%d %H%M%S",
    )
    df = df.set_index("datetime")
    df = df.loc["2026-01-05 00:00":"2026-01-05 00:10"]
    dfs[t] = df

# Merge on datetime using outer join, forward-fill missing values
df = pd.concat([dfs["ask"], dfs["bid"], dfs["last"]], axis=1).sort_index().ffill()
df["mid"] = (df["ask"] + df["bid"]) / 2
print(df.head(10))

fig, axes = plt.subplots(4, 1, figsize=(12, 24), sharex=True)
axes[0].plot(df.index, df["ask"], label="Ask", alpha=0.8)
axes[0].plot(df.index, df["bid"], label="Bid", alpha=0.8)
axes[0].plot(df.index, df["last"], label="Last", alpha=0.8)
axes[0].plot(df.index, df["mid"], label="Mid (Ask+Bid)/2", alpha=0.8, linestyle="--")
axes[0].set_title("EUR/USD — Raw Ticks: Ask, Bid, Last & Mid")
axes[0].set_ylabel("Price")
axes[0].legend()
for i, t in enumerate(["ask", "bid", "last"]):
    ax = axes[i + 1]
    ax.plot(df.index, df[t], label=t.capitalize(), color=f"C{i}")
    ax.set_title(f"EUR/USD — {t.capitalize()} (Raw Ticks)")
    ax.set_ylabel("Price")
plt.tight_layout()
fig.savefig("../plots/eda/eurusd_ask_bid_last_raw.pdf", bbox_inches="tight")
print("Saved!")
plt.show()